# 09 — Kaggle: LoRA (r16) + ControlNet + IP-Adapter — Inference & Metrics

**Mục tiêu:** Chạy inference trên Kaggle với weights LoRA đã train (notebook `08_lora_cn_ipa_local.ipynb`), đánh giá đầy đủ các metric giống notebook 08: **L1, L2, ICP, SS** (Yan et al.), **PSNR, SSIM, LPIPS, FID**.

**Input Kaggle:**
- Dataset xe: `/kaggle/input/datasets/dangvy1507/vehicle`
- LoRA weights: `/kaggle/input/datasets/dangvy1507/lora-weights` (ưu tiên thư mục `best/`, không có thì `final/`)

**Tập test:** Giống hệt notebook 08 (`CELL 3`) — trong toàn bộ CSV lấy **666 ảnh / bin** × 3 khoảng che [(0.2,0.4), (0.4,0.6), (0.6,0.8)], `random_state=42`, shuffle lại một lần → **đúng 1998 ảnh** nếu mỗi bin có đủ ≥666 ảnh và dữ liệu cùng phân phối như train local.

**Cấu trúc thư mục dataset (đề xuất, giống `synthetic_occ`):** `{vehicle}/metadata_synthetic_occ.csv` cùng cấp với các thư mục `x_gt`, `x_occ`, `masks` — hoặc `{vehicle}/synthetic_occ/...`. Notebook sẽ tự tìm CSV.

**LoRA trên Kaggle:** Thư mục `best/` hoặc `final/` chứa `adapter_model.safetensors` và `adapter_config.json` như sau khi `save_pretrained()` từ notebook 08; hoặc đặt trực tiếp các file adapter vào root dataset.

**Kaggle:** Bật **GPU Accelerator** và **Internet** (để Hugging Face tải SD1.5, ControlNet, IP-Adapter). Nếu cần, thêm Hugging Face token trong **Secrets** và `hf auth login`.

In [ ]:
# ── CELL 1 — Dependencies ────────────────────────────────────────────────
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers>=0.27.0",
        "peft>=0.8.0",
        "accelerate>=0.26.0",
        "transformers>=4.36.0",
        "xformers",
        "scikit-image",
        "lpips",
        "clean-fid",
        "opencv-python-headless",
        "huggingface_hub",
        "tqdm",
    ],
    check=True,
)
print("Dependencies ready")

In [ ]:
# ── CELL 2 — Config, paths Kaggle, load metadata ─────────────────────────
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path("/kaggle/input/datasets/dangvy1507/vehicle")
LORA_ROOT = Path("/kaggle/input/datasets/dangvy1507/lora-weights")

_meta_candidates = [
    DATA_ROOT / "metadata_synthetic_occ.csv",
    DATA_ROOT / "synthetic_occ" / "metadata_synthetic_occ.csv",
]
META_CSV = next((p for p in _meta_candidates if p.exists()), None)
if META_CSV is None:
    hits = sorted(DATA_ROOT.rglob("metadata_synthetic_occ.csv"))
    META_CSV = hits[0] if hits else None

assert META_CSV is not None and META_CSV.is_file(), (
    "Không tìm thấy metadata_synthetic_occ.csv dưới " + str(DATA_ROOT)
)

SYNTH_DIR = META_CSV.parent
GT_DIR = SYNTH_DIR / "x_gt"
OCC_DIR = SYNTH_DIR / "x_occ"
MASK_DIR = SYNTH_DIR / "masks"

OUT_BASE = Path("/kaggle/working/eval_lora_cn_ipa_kaggle")
PRED_DIR = OUT_BASE / "x_hat"
EVAL_GT_DIR = OUT_BASE / "fid_gt"
EVAL_PRED_DIR = OUT_BASE / "fid_pred"
REPORT_DIR = OUT_BASE / "reports"
for d in (PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

SD_MODEL_ID = "runwayml/stable-diffusion-inpainting"
CN_MODEL_ID = "lllyasviel/control_v11p_sd15_canny"
IP_REPO = "h94/IP-Adapter"
IP_WEIGHT = "models/ip-adapter-plus_sd15.bin"

LORA_RANK = 16
PROMPT = "a car, realistic, high quality, detailed, complete, no occlusion"
NEG_PROMPT = "blurry, distorted, artifacts, extra car, duplicate"
NUM_STEPS = 20
GUIDANCE = 7.5

# Grid search như notebook 08 / full-eval — True = dùng combo đã chứng minh
SKIP_GRID_SEARCH = True
BEST_CN = 0.60
BEST_IP = 0.50
CN_SCALE_GRID = [0.15, 0.30, 0.45, 0.60, 0.75]
IP_SCALE_GRID = [0.3, 0.5, 0.7]
GRID_SAMPLE = 40

CANNY_LOW = 80
CANNY_HIGH = 150
MASK_DILATE_PX = 5

BIN_EDGES = [(0.20, 0.40), (0.40, 0.60), (0.60, 0.80)]
BIN_LABELS = ["20-40%", "40-60%", "60-80%"]

meta = pd.read_csv(META_CSV)
meta.columns = meta.columns.str.strip().str.lower()
if "stem" not in meta.columns and "filename" in meta.columns:
    meta["stem"] = meta["filename"].str.replace(r"\.[^.]+$", "", regex=True)

def mask_path_for_row(row):
    m = row.get("mask")
    if m is None or (isinstance(m, float) and np.isnan(m)):
        pass
    elif isinstance(m, str) and len(m.strip()) > 0:
        return m.strip()
    return f"{row['stem']}.png"

if "mask" not in meta.columns:
    meta["mask"] = [mask_path_for_row(r) for _, r in meta.iterrows()]
else:
    meta["mask"] = [
        mask_path_for_row(meta.loc[i])
        if (pd.isna(meta.loc[i, "mask"]) or str(meta.loc[i, "mask"]).strip() == "")
        else str(meta.loc[i, "mask"]).strip()
        for i in meta.index
    ]

meta_full = meta.reset_index(drop=True)
assert (
    "occlusion_ratio" in meta_full.columns
), "CSV cần cột occlusion_ratio để lấy tập test giống notebook 08"

# Test giống notebook 08 CELL 3: 666/bin × 3 bins, seed=42
_per_bin = 2000 // len(BIN_EDGES)
_bin_dfs = []
for lo, hi in BIN_EDGES:
    sub = meta_full[
        (meta_full["occlusion_ratio"] >= lo) & (meta_full["occlusion_ratio"] < hi)
    ]
    _bin_dfs.append(sub.sample(min(_per_bin, len(sub)), random_state=SEED))
eval_df = (
    pd.concat(_bin_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)
)

def _occ_bin_label(ratio):
    for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
        if lo <= ratio < hi:
            return lbl
    return "?"

eval_df["bin_label"] = eval_df["occlusion_ratio"].map(_occ_bin_label)

_lora_candidates = [
    LORA_ROOT / "best",
    LORA_ROOT / "final",
]
best_ckpt = next((p for p in _lora_candidates if p.is_dir()), None)
if best_ckpt is None:
    # Nếu user zip thẳng nội dung adapter vào root
    if (LORA_ROOT / "adapter_model.safetensors").is_file():
        best_ckpt = LORA_ROOT

assert best_ckpt is not None and best_ckpt.is_dir(), (
    "Không tìm thấy LoRA (best/, final/, hoặc adapter ở root): " + str(LORA_ROOT)
)

print("META_CSV  :", META_CSV)
print("SYNTH_DIR :", SYNTH_DIR)
print("Dataset   :", len(meta_full), "rows CSV | test (NB08):", len(eval_df))
if len(eval_df) != 1998:
    print(
        "WARN: NB08 kỳ vọng 1998 ảnh (666/bin); thực tế",
        len(eval_df),
        "— bin có thể thiếu ảnh hoặc CSV khác bộ train local.",
    )
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    n = len(
        eval_df[(eval_df["occlusion_ratio"] >= lo) & (eval_df["occlusion_ratio"] < hi)]
    )
    print(f"  test {lbl}: {n} ảnh")
print("LoRA ckpt :", best_ckpt)
print("DEVICE    :", DEVICE, "| dtype", DTYPE)

In [ ]:
# ── CELL 3 — SD + ControlNet + IP-Adapter + merge LoRA ───────────────────
import cv2
from PIL import Image

from diffusers import (
    ControlNetModel,
    DPMSolverMultistepScheduler,
    StableDiffusionControlNetInpaintPipeline,
)
from peft import PeftModel

print("Loading ControlNet ...")
controlnet = ControlNetModel.from_pretrained(CN_MODEL_ID, torch_dtype=DTYPE)

print("Loading SD1.5 Inpainting ...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    SD_MODEL_ID,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

print(f"Merging LoRA from {best_ckpt} ...")
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(best_ckpt))
pipe.unet = pipe.unet.merge_and_unload()
print("LoRA merged into UNet")

pipe = pipe.to(DEVICE)
xformers_ok = False
try:
    pipe.enable_xformers_memory_efficient_attention()
    xformers_ok = True
    print("xFormers: OK")
except Exception:
    pipe.enable_attention_slicing("auto")
    print("xFormers: fallback → attention slicing")
pipe.vae.enable_slicing()

use_ip = False
try:
    pipe.load_ip_adapter(IP_REPO, subfolder="models", weight_name=IP_WEIGHT)
    use_ip = True
    print("IP-Adapter: OK")
except Exception as e:
    print("IP-Adapter: FAILED", e)

print(
    "Pipeline OK | LoRA r=",
    LORA_RANK,
    "| IP-Adapter=",
    use_ip,
    "| xFormers=",
    xformers_ok,
)

In [ ]:
# ── CELL 4 — Infer helpers ────────────────────────────────────────────────


def extract_canny_masked(
    image_pil,
    mask_pil,
    low=CANNY_LOW,
    high=CANNY_HIGH,
    dilate_px=MASK_DILATE_PX,
):
    img = np.array(image_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    mask_np = np.array(mask_pil.convert("L"))
    if dilate_px > 0:
        k = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (dilate_px * 2 + 1, dilate_px * 2 + 1),
        )
        mask_np = cv2.dilate(mask_np, k, iterations=1)
    edges[mask_np > 127] = 0
    return Image.fromarray(np.stack([edges] * 3, axis=2))


def extract_visible_patch(image_pil, mask_pil):
    img = np.array(image_pil.convert("RGB"))
    mask = np.array(mask_pil.convert("L"))
    vis = img.copy()
    vis[mask > 127] = 128
    return Image.fromarray(vis)


def inpaint(image, mask, prompt, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED):
    canny = extract_canny_masked(image, mask)
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    vis_ref = extract_visible_patch(image, mask)
    call_kwargs = dict(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        image=image,
        control_image=canny,
        mask_image=mask,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=cn_scale,
        generator=gen,
    )
    if use_ip:
        pipe.set_ip_adapter_scale(ip_scale)
        call_kwargs["ip_adapter_image"] = vis_ref
    ctx = torch.autocast("cuda") if DEVICE == "cuda" else torch.no_grad()
    with ctx:
        result = pipe(**call_kwargs).images[0]
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result


print("Helpers ready.")

In [ ]:
# ── CELL 5 — Preview 4 mẫu ────────────────────────────────────────────────
import matplotlib.pyplot as plt

samples_vis = eval_df.sample(min(4, len(eval_df)), random_state=SEED).reset_index(
    drop=True
)
n_cols = 5 if use_ip else 4
fig, axes = plt.subplots(len(samples_vis), n_cols, figsize=(4 * n_cols, 3.5 * len(samples_vis)))
if len(samples_vis) == 1:
    axes = np.array([axes])

for i, row in samples_vis.iterrows():
    occ = Image.open(OCC_DIR / row["x_occ"]).convert("RGB").resize((512, 512))
    msk = Image.open(MASK_DIR / row["mask"]).convert("L").resize((512, 512))
    gt = Image.open(GT_DIR / row["x_gt"]).convert("RGB").resize((512, 512))
    can = extract_canny_masked(occ, msk)
    col = 0

    def show(ax, img, title, cmap=None):
        ax.imshow(img, cmap=cmap)
        if i == 0:
            ax.set_title(title, fontsize=9, fontweight="bold")
        ax.axis("off")

    show(axes[i, col], occ, "x_occ")
    col += 1
    show(axes[i, col], msk, "Mask", "gray")
    col += 1
    show(axes[i, col], can, "Canny")
    col += 1
    if use_ip:
        show(axes[i, col], extract_visible_patch(occ, msk), "Visible ref")
        col += 1
    show(axes[i, col], gt, "x_gt")
    occ_r = float(row.get("occlusion_ratio", 0.0))
    axes[i, 0].set_ylabel(f"occ={occ_r:.2f}", fontsize=8)

plt.suptitle("Preview — LoRA + CN + IP-Adapter", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 6 — Grid search (tùy chọn) + metric helpers ──────────────────────
from tqdm.auto import tqdm
import lpips as lpips_lib
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

lpips_model = lpips_lib.LPIPS(net="alex").to(DEVICE).eval()


def pixel_l1(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m = mask01.astype(bool)
    return (
        float(np.mean(np.abs(gt_f[m] - pr_f[m])))
        if m.sum() > 0
        else float(np.mean(np.abs(gt_f - pr_f)))
    )


def pixel_l2(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m = mask01.astype(bool)
    return (
        float(np.mean((gt_f[m] - pr_f[m]) ** 2))
        if m.sum() > 0
        else float(np.mean((gt_f - pr_f) ** 2))
    )


def masked_psnr(gt, pred, mask01):
    m = mask01.astype(bool)
    if m.sum() == 0:
        return float(psnr_fn(gt, pred, data_range=255))
    gt_m = gt[m].astype(np.float64)
    pr_m = pred[m].astype(np.float64)
    mse = np.mean((gt_m - pr_m) ** 2)
    if mse == 0:
        return 100.0
    return float(10 * np.log10(255.0**2 / mse))


def masked_ssim(gt, pred, mask01):
    _, smap = ssim_fn(
        gt.astype(np.float32) / 255.0,
        pred.astype(np.float32) / 255.0,
        channel_axis=2,
        data_range=1.0,
        full=True,
    )
    m = mask01.astype(bool)
    return float(smap[m].mean()) if m.sum() > 0 else float(smap.mean())


def masked_lpips(gt, pred, mask01):
    m = mask01.astype(np.float32)[..., None]
    gt_t = torch.from_numpy((gt * m).astype(np.uint8)).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    pr_t = torch.from_numpy((pred * m).astype(np.uint8)).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())


if SKIP_GRID_SEARCH:
    print(
        "SKIP_GRID_SEARCH=True — dùng BEST_CN=",
        BEST_CN,
        "BEST_IP=",
        BEST_IP,
    )
    grid_df = pd.DataFrame(
        [
            {
                "cn_scale": BEST_CN,
                "ip_scale": BEST_IP,
                "ssim_mean": None,
                "lpips_mean": None,
                "psnr_mean": None,
            }
        ]
    )
else:
    grid_sample = eval_df.sample(
        min(GRID_SAMPLE, len(eval_df)), random_state=SEED
    ).reset_index(drop=True)
    ip_values = IP_SCALE_GRID if use_ip else [0.0]
    grid_res = []
    for cn_s in CN_SCALE_GRID:
        for ip_s in ip_values:
            ss, lp, ps = [], [], []
            for _, r in tqdm(
                grid_sample.iterrows(),
                total=len(grid_sample),
                desc=f"cn={cn_s} ip={ip_s}",
            ):
                occ = Image.open(OCC_DIR / r["x_occ"]).convert("RGB").resize((512, 512))
                msk = Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512))
                gt = Image.open(GT_DIR / r["x_gt"]).convert("RGB").resize((512, 512))
                pred = inpaint(occ, msk, PROMPT, cn_scale=cn_s, ip_scale=ip_s)
                gt_np = np.array(gt)
                pr_np = np.array(pred)
                mk01 = (np.array(msk) > 127).astype(np.uint8)
                ss.append(masked_ssim(gt_np, pr_np, mk01))
                lp.append(masked_lpips(gt_np, pr_np, mk01))
                ps.append(masked_psnr(gt_np, pr_np, mk01))
            row_r = {
                "cn_scale": cn_s,
                "ip_scale": ip_s,
                "ssim_mean": float(np.mean(ss)),
                "lpips_mean": float(np.mean(lp)),
                "psnr_mean": float(np.mean(ps)),
            }
            grid_res.append(row_r)
            print(
                f"  cn={cn_s} ip={ip_s} | PSNR={row_r['psnr_mean']:.2f} LPIPS={row_r['lpips_mean']:.4f}"
            )
    grid_df = pd.DataFrame(grid_res)
    best_idx = int(grid_df["lpips_mean"].idxmin())
    BEST_CN = float(grid_df.loc[best_idx, "cn_scale"])
    BEST_IP = float(grid_df.loc[best_idx, "ip_scale"])
    grid_df.to_csv(REPORT_DIR / "grid_search.csv", index=False)
    print("Best combo:", BEST_CN, BEST_IP)

print("Using cn_scale=", BEST_CN, "ip_scale=", BEST_IP)

In [ ]:
# ── CELL 7 — Full inference ─────────────────────────────────────────────────
import time

from tqdm.auto import tqdm

print(
    "Inference:",
    len(eval_df),
    "images | cn=",
    BEST_CN,
    "ip=",
    BEST_IP,
    "steps=",
    NUM_STEPS,
)
pred_records = []
t0 = time.time()

for _, r in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Inference"):
    occ = Image.open(OCC_DIR / r["x_occ"]).convert("RGB").resize((512, 512))
    msk = Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512))
    gt = Image.open(GT_DIR / r["x_gt"]).convert("RGB").resize((512, 512))
    pred = inpaint(occ, msk, PROMPT, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED)
    fname = f"{r['stem']}.png"
    pred.save(PRED_DIR / fname)
    gt.save(EVAL_GT_DIR / fname)
    pred.save(EVAL_PRED_DIR / fname)
    pred_records.append(
        {
            "stem": r["stem"],
            "x_gt": r["x_gt"],
            "x_occ": r["x_occ"],
            "mask": r["mask"],
            "occlusion_ratio": float(r.get("occlusion_ratio", 0.0)),
            "bin_label": r.get("bin_label", "?"),
            "pred": fname,
        }
    )

elapsed = time.time() - t0
pred_df = pd.DataFrame(pred_records)
pred_df.to_csv(REPORT_DIR / "pred_index.csv", index=False)
n = max(len(pred_df), 1)
print(
    "Done:",
    len(pred_df),
    "images |",
    f"{elapsed / 60:.1f} min | ~{elapsed / n:.1f}s/img",
)

In [ ]:
# ── CELL 8 — Per-image metrics + FID (L1,L2,ICP,SS,PSNR,SSIM,LPIPS)
from cleanfid import fid as cleanfid
from torchvision import transforms
from torchvision.models import inception_v3
from torchvision.models.segmentation import deeplabv3_resnet101
from tqdm.auto import tqdm

print("Loading Inception V3 (ICP) ...")
try:
    from torchvision.models import Inception_V3_Weights

    inception_net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except (ImportError, AttributeError):
    inception_net = inception_v3(pretrained=True)
inception_net.eval().to(DEVICE)

IMAGENET_CAR_CLASSES = [407, 436, 511, 627, 656, 705, 717, 734, 751, 779, 817, 820, 868]
inception_tf = transforms.Compose(
    [
        transforms.Resize(299),
        transforms.CenterCrop(299),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def compute_icp(pred_pil, mask_np_gray):
    ys, xs = np.where(mask_np_gray > 127)
    if len(ys) == 0:
        return 0.0
    pad = 16
    h, w = mask_np_gray.shape
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(h, int(ys.max()) + pad)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(w, int(xs.max()) + pad)
    crop = pred_pil.crop((x0, y0, x1, y1))
    if crop.width < 10 or crop.height < 10:
        return 0.0
    inp = inception_tf(crop.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = inception_net(inp)
        logits = out.logits if hasattr(out, "logits") else out
        probs = torch.softmax(logits, dim=1)[0].cpu()
    return float(probs[IMAGENET_CAR_CLASSES].sum())


print("Loading DeepLab V3 (SS) ...")
try:
    from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights

    deeplab_net = deeplabv3_resnet101(
        weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1
    )
except (ImportError, AttributeError):
    deeplab_net = deeplabv3_resnet101(pretrained=True)
deeplab_net.eval().to(DEVICE)
CAR_CLASS_IDX = 7
seg_tf = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def compute_ss(pred_pil, mask01):
    inp = seg_tf(pred_pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        seg_out = deeplab_net(inp)["out"][0]
    pred_seg = seg_out.argmax(0).cpu().numpy().astype(np.uint8)
    pred_seg = cv2.resize(pred_seg, (512, 512), interpolation=cv2.INTER_NEAREST)
    m = mask01.astype(bool)
    return float((pred_seg[m] == CAR_CLASS_IDX).mean()) if m.sum() > 0 else 0.0


metric_rows = []
for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Metrics"):
    gt_bgr = cv2.imread(str(GT_DIR / r["x_gt"]))
    pr_bgr = cv2.imread(str(PRED_DIR / r["pred"]))
    if gt_bgr is None or pr_bgr is None:
        continue
    gt_np = cv2.resize(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    pr_np = cv2.resize(cv2.cvtColor(pr_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    mk_raw = cv2.imread(str(MASK_DIR / r["mask"]), cv2.IMREAD_GRAYSCALE)
    mk_np = cv2.resize(mk_raw, (512, 512))
    mk01 = (mk_np > 127).astype(np.uint8)
    pr_pil = Image.fromarray(pr_np)
    metric_rows.append(
        {
            "stem": r["stem"],
            "occlusion_ratio": r["occlusion_ratio"],
            "bin_label": r.get("bin_label", "?"),
            "l1": pixel_l1(gt_np, pr_np, mk01),
            "l2": pixel_l2(gt_np, pr_np, mk01),
            "icp": compute_icp(pr_pil, mk_np),
            "ss": compute_ss(pr_pil, mk01),
            "psnr": masked_psnr(gt_np, pr_np, mk01),
            "ssim": masked_ssim(gt_np, pr_np, mk01),
            "lpips": masked_lpips(gt_np, pr_np, mk01),
        }
    )

metric_df = pd.DataFrame(metric_rows)
print("Computing FID (clean)...")
fid_value = cleanfid.compute_fid(str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode="clean")
print("FID =", round(fid_value, 2))

metric_df.to_csv(REPORT_DIR / "metrics_per_image.csv", index=False)
print(metric_df[["l1", "l2", "icp", "ss", "psnr", "ssim", "lpips"]].describe().round(4))

In [ ]:
# ── CELL 9 — Summary: overall + theo bin + lưu CSV/JSON ───────────────────
ALL_METRICS = ["l1", "l2", "icp", "ss", "psnr", "ssim", "lpips"]


def bin_summary(df, lo, hi, label):
    sub = df[(df["occlusion_ratio"] >= lo) & (df["occlusion_ratio"] < hi)]
    if len(sub) == 0:
        return {"bin": label, "n": 0, **{m: None for m in ALL_METRICS}}
    row = {"bin": label, "n": len(sub)}
    for m in ALL_METRICS:
        row[m] = round(float(sub[m].mean()), 4)
    return row


overall = {"bin": "overall", "n": len(metric_df)}
for m in ALL_METRICS:
    overall[m] = round(float(metric_df[m].mean()), 4)

rows = [overall]
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    rows.append(bin_summary(metric_df, lo, hi, lbl))

summary_df = pd.DataFrame(rows)
summary_df.insert(
    summary_df.columns.get_loc("psnr") + 3,
    "fid",
    [round(fid_value, 2)] + [None] * len(BIN_EDGES),
)

print("=" * 80)
print("KAGGLE RESULTS — SD1.5 + LoRA r=" + str(LORA_RANK) + " + CN + IP-Adapter")
print(
    "cn_scale=",
    BEST_CN,
    "ip_scale=",
    BEST_IP,
    "steps=",
    NUM_STEPS,
    "n=",
    len(metric_df),
)
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

summary_df.to_csv(REPORT_DIR / "summary_table.csv", index=False)
with open(REPORT_DIR / "eval_config.json", "w") as f:
    json.dump(
        {
            "config": f"SD1.5+LoRA r={LORA_RANK}+CN+IPA Kaggle",
            "lora_ckpt": str(best_ckpt),
            "data_csv": str(META_CSV),
            "cn_scale": BEST_CN,
            "ip_scale": BEST_IP,
            "num_steps": NUM_STEPS,
            "n": int(len(metric_df)),
            **{m: overall.get(m) for m in ALL_METRICS},
            "fid": round(float(fid_value), 2),
        },
        f,
        indent=2,
    )

baseline_vals = {
    "l1": 0.1438,
    "l2": 0.0521,
    "icp": 0.6081,
    "ss": 0.3319,
    "psnr": 13.5475,
    "ssim": 0.4151,
    "lpips": 0.1285,
}
print("\nSo sánh nhanh với SD1.5+CN+IPA (không LoRA) từ full-eval (hardcoded):")
cmp_rows = [
    {"config": "SD1.5+CN+IPA (no LoRA)", **{m: round(baseline_vals[m], 4) for m in ALL_METRICS}},
    {
        "config": f"SD1.5+LoRA r={LORA_RANK}+CN+IPA (Kaggle eval)",
        **{m: overall.get(m) for m in ALL_METRICS},
    },
]
cmp_df = pd.DataFrame(cmp_rows)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(REPORT_DIR / "lora_vs_baseline.csv", index=False)

In [ ]:
# ── CELL 10 — Visualization + zip báo cáo ─────────────────────────────────
import zipfile

import matplotlib.pyplot as plt

plot_df = summary_df[summary_df["bin"] != "overall"].copy()
if len(plot_df) > 0 and plot_df["n"].sum() > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col, label in [
        (axes[0], "psnr", "PSNR (dB)"),
        (axes[1], "ssim", "SSIM"),
        (axes[2], "lpips", "LPIPS"),
    ]:
        sub = plot_df.dropna(subset=[col])
        if len(sub) == 0:
            continue
        colors = ["#3498db", "#2ecc71", "#e74c3c"]
        bars = ax.bar(sub["bin"], sub[col], color=colors[: len(sub)], width=0.5)
        ax.set_title(label, fontweight="bold")
        ax.set_xlabel("Occlusion bin")
        for bar, v in zip(bars, sub[col]):
            if v is not None:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.002,
                    f"{v:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                )
    plt.suptitle("Metrics theo bin — LoRA + CN + IPA", y=1.02)
    plt.tight_layout()
    plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
sc = ax2.scatter(
    metric_df["occlusion_ratio"],
    metric_df["psnr"],
    c=metric_df["psnr"],
    cmap="RdYlGn",
    alpha=0.35,
    s=8,
)
plt.colorbar(sc, ax=ax2, label="PSNR (dB)")
for lo, hi in BIN_EDGES:
    ax2.axvline(lo, color="gray", linestyle="--", linewidth=0.8)
ax2.axvline(BIN_EDGES[-1][1], color="gray", linestyle="--", linewidth=0.8)
ax2.set_xlabel("Occlusion ratio")
ax2.set_ylabel("PSNR (dB)")
ax2.set_title("PSNR vs occlusion ratio")
plt.tight_layout()
plt.show()

zip_path = OUT_BASE / "kaggle_lora_reports.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
    for f in sorted(REPORT_DIR.glob("*")):
        zf.write(f, f"reports/{f.name}")
print("Zip:", zip_path, "— files:", [f.name for f in sorted(REPORT_DIR.glob("*"))])